# Churn: fit and visualise a decision tree

*Session 10, block 1 practice. Author: course team, licence CC-BY-4.0.*

1. compute the best first split by hand with the Gini impurity,
2. fit trees of increasing depth and compare training and cross-validated scores,
3. prune with `ccp_alpha`,
4. draw and read the final tree.

Theory: [01-decision-trees.md](../theory/01-decision-trees.md). The data are read from GitHub.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
cat = X.select_dtypes(exclude="number").columns.tolist()
num = X.select_dtypes("number").columns.tolist()

X_oh = pd.get_dummies(X, columns=cat, drop_first=True, dtype=int)
X_tr, X_te, y_tr, y_te = train_test_split(X_oh, y, test_size=0.25, stratify=y, random_state=0)
print(X_tr.shape, round(y_tr.mean(), 3))

## 1. The best first split by hand

For every candidate threshold of `tenure`, compute the weighted Gini impurity of the two children on the training data.

In [ ]:
import matplotlib.pyplot as plt


def gini(yy):
    p = np.bincount(yy, minlength=2) / len(yy)
    return 1 - (p ** 2).sum()


def split_gini(mask, yy):
    return mask.mean() * gini(yy[mask]) + (~mask).mean() * gini(yy[~mask])


yy = y_tr.to_numpy()
values = np.sort(X_tr["tenure"].unique())
thresholds = (values[:-1] + values[1:]) / 2                  # midpoints, as scikit-learn uses
scores = [split_gini(X_tr["tenure"].to_numpy() <= t, yy) for t in thresholds]
best = thresholds[int(np.argmin(scores))]
print("root impurity", round(gini(yy), 3), "| best tenure threshold", best, "| weighted impurity", round(min(scores), 3))

plt.plot(thresholds, scores)
plt.axvline(best, ls=":", color="grey")
plt.xlabel("tenure threshold")
plt.ylabel("weighted Gini impurity")
plt.title("Weighted impurity of the split 'tenure <= t'");

**Task 1.** Fit `DecisionTreeClassifier(max_depth=1)` on `X_tr[["tenure"]]` and check that it chooses the same threshold. Then fit it on all features: is `tenure` still the first split? Compare the weighted impurity of the best `Contract_Two year` split with the tenure split.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

stump = DecisionTreeClassifier(max_depth=1).fit(X_tr[["tenure"]], y_tr)
print(export_text(stump, feature_names=["tenure"]))
# TODO (Task 1): the same with all features


## 2. Depth and overfitting

In [ ]:
from sklearn.model_selection import cross_val_score

rows = []
for depth in range(1, 21):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    cv_acc = cross_val_score(tree, X_tr, y_tr, cv=5).mean()
    tree.fit(X_tr, y_tr)
    rows.append({"depth": depth, "leaves": tree.get_n_leaves(), "train": tree.score(X_tr, y_tr), "cv": cv_acc})
curve = pd.DataFrame(rows).set_index("depth")
curve[["train", "cv"]].plot(marker="o", ylabel="accuracy", title="Training vs cross-validated accuracy")
curve.round(3).head(10)

**Task 2.** Repeat the curve with `scoring="roc_auc"` and with `min_samples_leaf=20`. How do the curves change, and why does `min_samples_leaf` slow down overfitting?

In [ ]:
# TODO (Task 2)


## 3. Pruning with cost-complexity

In [ ]:
from sklearn.model_selection import GridSearchCV

path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_tr, y_tr)
alphas = np.unique(np.quantile(path.ccp_alphas, np.linspace(0, 0.995, 40)))   # 40 candidate values
search = GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alphas}, cv=5,
                      scoring="roc_auc").fit(X_tr, y_tr)
pruned = search.best_estimator_
print("best alpha", round(search.best_params_["ccp_alpha"], 5), "| CV AUC", round(search.best_score_, 3),
      "| leaves", pruned.get_n_leaves(), "| depth", pruned.get_depth())

res = pd.DataFrame(search.cv_results_)
plt.semilogx(res["param_ccp_alpha"].astype(float), res["mean_test_score"], marker="o")
plt.xlabel("ccp_alpha")
plt.ylabel("CV ROC AUC");

## 4. Draw and read the final tree

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.tree import plot_tree

print("test ROC AUC of the pruned tree:", round(roc_auc_score(y_te, pruned.predict_proba(X_te)[:, 1]), 3))
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(pruned, feature_names=list(X_tr.columns), class_names=["stays", "churns"], filled=True,
          max_depth=3, fontsize=8, ax=ax);

**Task 3.** Write down the two leaves with the highest churn rate as if-then rules a retention team could use. How many customers fall into each?

**Task 4 (optional).** Fit the pruned tree on 10 different bootstrap samples of the training data and print the first two splits each time. What does this tell you about the stability of single trees, and which method of page 2 uses this instability?

In [ ]:
# TODO (Tasks 3 and 4)
